# VoiceMOS figures with LaTeX typography

Colab-ready versions of all four analyses in `figures/`. Styling follows
`Results_Latex_seeingculture.ipynb`: LaTeX-rendered serif text, bold axis labels
and titles, 14 pt axis/tick text, and 11 pt legends. No GPU is needed.

**In Google Colab:**

1. Open this notebook and run the setup cells. They install the LaTeX packages
   in the Colab runtime, not on the cluster.
2. Run the repository cell to clone `McGill-NLP/VoiceMOS-Challenge-2026`
   and check out `dev.dg/figures`. The plotting CSVs are already in the branch.
3. Run any figure section after setup. Sections load their own data and can be
   rerun independently. The last cell downloads the generated figures as a ZIP.

The plots read the CSV tables from the checkout; no audio, model loading, or
training is required. When running locally, the notebook uses the current
repository checkout (which must be on `dev.dg/figures`). `USE_TEX = False` permits a local preview without
LaTeX, but the default is **True** for the final Colab exports.

The first two sections contain the preferred figures. Human-versus-model uses
scatter points and has **no statistics below or inside its panels**; its numbers
are shown separately in the notebook for use in the paper's LaTeX text.

## Setup: Colab dependencies and LaTeX

In [ ]:
from pathlib import Path
import importlib.util
import os
import shutil
import subprocess
import sys

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

USE_TEX = True
INSTALL_COLAB_DEPENDENCIES = True

if IN_COLAB and INSTALL_COLAB_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'matplotlib>=3.6', 'numpy>=1.23', 'scipy>=1.10'], check=True)
    if USE_TEX:
        print('Installing LaTeX packages in the Colab runtime...')
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run([
            'apt-get', 'install', '-y', '-qq',
            'texlive-latex-recommended', 'texlive-latex-extra',
            'texlive-fonts-recommended', 'texlive-plain-generic',
            'cm-super', 'dvipng',
        ], check=True, env={**os.environ, 'DEBIAN_FRONTEND': 'noninteractive'})

if USE_TEX:
    missing = [program for program in ['latex', 'dvipng'] if not shutil.which(program)]
    if missing:
        raise RuntimeError(
            f'Missing LaTeX tools: {missing}. Run the setup in Colab, or set '
            'USE_TEX = False for a local preview.'
        )
print(f'Colab: {IN_COLAB}; LaTeX rendering: {USE_TEX}')

### Clone the repository and load the plotting CSVs

In Colab, clone [VoiceMOS-Challenge-2026](https://github.com/McGill-NLP/VoiceMOS-Challenge-2026.git)
into `/content/VoiceMOS-Challenge-2026` on branch `dev.dg/figures`.
Rerunning this cell fetches the branch and updates the Colab checkout with a
fast-forward merge. It does not discard local edits.

The four plotting tables are read directly from `figures/<analysis>/<analysis>.csv`.
No CSV upload is needed. Locally, this cell reuses the current repository without
fetching or switching branches; check out `dev.dg/figures` before running it.

In [ ]:
REPO_URL = 'https://github.com/McGill-NLP/VoiceMOS-Challenge-2026.git'
BRANCH = 'dev.dg/figures'
ANALYSES = ('human_vs_model', 'listener_similarity',
            'listener_disagreement', 'attribute_differences')

if IN_COLAB:
    REPO_DIR = Path('/content/VoiceMOS-Challenge-2026')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                        REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
        subprocess.run(['git', '-C', str(REPO_DIR), 'merge', '--ff-only', f'origin/{BRANCH}'], check=True)
else:
    REPO_DIR = Path(subprocess.check_output(
        ['git', 'rev-parse', '--show-toplevel'], text=True).strip())

active_branch = subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'branch', '--show-current'], text=True).strip()
if active_branch != BRANCH:
    raise RuntimeError(f'Expected {BRANCH}, found {active_branch}. Run git switch {BRANCH} first.')
os.chdir(REPO_DIR)
DATA_DIR = REPO_DIR / 'figures'


def find_csv(name):
    path = DATA_DIR / name / f'{name}.csv'
    return path if path.is_file() else None


missing = [name for name in ANALYSES if find_csv(name) is None]
if missing:
    raise FileNotFoundError(f'Missing plotting CSVs on {BRANCH}: {missing}')
print(f'Repository: {REPO_DIR} (branch: {active_branch})')
for name in ANALYSES:
    print(f'{name}: {find_csv(name)}')

# Separate exports from the original figures. Each section gets its own folder.
OUTPUT_DIR = (Path('/content/voicemos_latex_figures') if IN_COLAB
              else DATA_DIR / 'latex_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Export directory:', OUTPUT_DIR)

### Shared typography and export helpers

Change the font-size constants here to adjust all figures together. Bold text
uses `	extbf{...}` when LaTeX is enabled, as in the reference notebook. PDF and
SVG exports retain vector graphics; PNG exports use 300 dpi.

In [ ]:
import csv
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.ticker import FuncFormatter
from scipy.stats import spearmanr
from IPython.display import display, Markdown, FileLink

LABEL_SIZE = 14
TICK_SIZE = 14
TITLE_SIZE = 14
LEGEND_SIZE = 11
ANNOTATION_SIZE = 11
SPEAKER_COLOR = '#0072B2'
ACCENT_COLOR = '#D55E00'

plt.rcParams.update({
    'text.usetex': USE_TEX,
    'font.family': 'serif',
    'font.size': 12,
    'axes.labelsize': LABEL_SIZE,
    'axes.labelweight': 'bold',
    'axes.titlesize': TITLE_SIZE,
    'axes.titleweight': 'bold',
    'xtick.labelsize': TICK_SIZE,
    'ytick.labelsize': TICK_SIZE,
    'legend.fontsize': LEGEND_SIZE,
    'axes.unicode_minus': False,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'path',
    'savefig.dpi': 300,
})


def bold(text):
    """Escape literal labels for LaTeX; support a non-LaTeX preview too."""
    text = str(text)
    if not USE_TEX:
        return text
    replacements = {'\\': r'\textbackslash{}', '&': r'\&', '%': r'\%',
                    '$': r'\$', '#': r'\#', '_': r'\_', '{': r'\{',
                    '}': r'\}', '~': r'\textasciitilde{}', '^': r'\textasciicircum{}'}
    escaped = ''.join(replacements.get(character, character) for character in text)
    return r'\textbf{' + escaped + '}'


def style_axes(ax, numeric_y=True):
    ax.tick_params(axis='both', which='major', labelsize=TICK_SIZE)
    formatter = FuncFormatter(lambda value, position: bold(f'{value:g}'))
    ax.xaxis.set_major_formatter(formatter)
    if numeric_y:
        ax.yaxis.set_major_formatter(formatter)
    ax.spines[['top', 'right']].set_visible(False)
    if not USE_TEX:
        for tick in ax.get_xticklabels() + ax.get_yticklabels():
            tick.set_fontweight('bold')


def load_rows(name, required):
    path = find_csv(name)
    if path is None:
        raise FileNotFoundError(f'Missing {name}.csv. Rerun the repository setup cell and check the branch.')
    with path.open(newline='', encoding='utf-8') as handle:
        reader = csv.DictReader(handle)
        missing = set(required) - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f'{path}: missing columns {sorted(missing)}')
        rows = list(reader)
    if not rows:
        raise ValueError(f'{path}: no rows')
    print(f'{name}: {len(rows):,} rows from {path.name}')
    return rows


def numbers(rows, column):
    values = np.array([float(row[column]) for row in rows])
    if not np.isfinite(values).all():
        raise ValueError(f'{column}: expected finite values')
    return values


def unique_pairs(rows):
    keys = [(row['wav_a_path'], row['wav_b_path']) for row in rows]
    if len(set(keys)) != len(keys):
        raise ValueError('Duplicate waveform pairs in plotting CSV')


def srcc(x, y):
    return float(spearmanr(x, y).statistic) if np.ptp(x) and np.ptp(y) else None


def show_table(headers, rows):
    lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---'] * len(headers)) + ' |']
    lines.extend('| ' + ' | '.join(map(str, row)) + ' |' for row in rows)
    display(Markdown('\n'.join(lines)))


def save_figure(fig, name, summary):
    folder = OUTPUT_DIR / name
    folder.mkdir(parents=True, exist_ok=True)
    for extension in ['pdf', 'png', 'svg']:
        path = folder / f'{name}.{extension}'
        fig.savefig(path, bbox_inches='tight', dpi=300)
        print(path)
    (folder / 'summary.json').write_text(json.dumps(summary, indent=2, allow_nan=False) + '\n')
    plt.show()
    plt.close(fig)


# Fail early if LaTeX is present but a required font/package is missing.
if USE_TEX:
    probe = plt.figure(figsize=(2, 1))
    probe.text(0.05, 0.5, bold('LaTeX font check: 50%'))
    try:
        probe.canvas.draw()
    finally:
        plt.close(probe)
print('Shared plotting style is ready.')

## 1. Human ratings versus model predictions

**Question:** Do human scores and model predictions distinguish speaker from accent?

Use the same **600 test pairs** in both panels. Human values are per-pair MOS;
model values come from the submitted weak-16 ensembles fitted on train+dev.
One point represents one pair, with equal axes, matching transparency, and an
equality line. No binning, jitter, color bar, or statistical annotations are added.

The statistics below are notebook output only, for use in the paper's LaTeX text.
Speaker--accent SRCC measures association between the two attributes, not
prediction accuracy. Greater association does not by itself show that a model
cannot distinguish them; the predicted score range is also narrower.

In [ ]:
human_rows = load_rows('human_vs_model', [
    'wav_a_path', 'wav_b_path', 'human_speaker', 'human_accent', 'model_speaker', 'model_accent'])
unique_pairs(human_rows)
human_values = np.column_stack([numbers(human_rows, 'human_speaker'), numbers(human_rows, 'human_accent')])
model_values = np.column_stack([numbers(human_rows, 'model_speaker'), numbers(human_rows, 'model_accent')])
for values in [human_values, model_values]:
    if ((values < 1) | (values > 5)).any():
        raise ValueError('Expected scores on the 1-5 scale')
human_summary = {'n_pairs': len(human_rows), 'split': 'test', 'model_fit': 'train+dev'}
for name, values in [('human', human_values), ('model', model_values)]:
    human_summary[name] = {
        'speaker_accent_srcc': srcc(values[:, 0], values[:, 1]),
        'mean_absolute_gap': float(np.abs(values[:, 1] - values[:, 0]).mean()),
    }
show_table(['Statistic', 'Human MOS', 'Weak-16 predictions'], [
    ['Speaker--accent SRCC', human_summary['human']['speaker_accent_srcc'], human_summary['model']['speaker_accent_srcc']],
    ['Mean absolute gap (MOS points)', human_summary['human']['mean_absolute_gap'], human_summary['model']['mean_absolute_gap']],
    ['Pairs', len(human_rows), len(human_rows)],
])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True, layout='constrained')
for ax, values, title in [
    (axes[0], human_values, '(a) Human mean ratings'),
    (axes[1], model_values, '(b) Submitted weak-16 predictions'),
]:
    ax.scatter(values[:, 0], values[:, 1], s=22, color=SPEAKER_COLOR,
               alpha=0.4, edgecolors='none', zorder=3)
    ax.plot([1, 5], [1, 5], '--', color='#888888', linewidth=1.5, zorder=2)
    ax.set(xlim=(0.95, 5.05), ylim=(0.95, 5.05), aspect='equal',
           xticks=[1, 2, 3, 4, 5], yticks=[1, 2, 3, 4, 5])
    ax.set_title(bold(title))
    ax.set_xlabel(bold('Speaker similarity'))
    style_axes(ax)
axes[0].set_ylabel(bold('Accent similarity'))
save_figure(fig, 'human_vs_model', human_summary)

## 2. How much do individual listeners distinguish speaker from accent?

All **25 training listeners**, using 13,687 individual rating rows. The plotting
CSV contains the statistics already computed from those ratings. The plot
orders listeners by decreasing speaker--accent Spearman correlation.
Labels show listener IDs only; rating counts remain in the statistics table.

Correlation measures similar ranking, not identical scores, and does not alone
establish confusion between the concepts. Identical-score percentages remain
in the statistics table but are not plotted.
Listener 21, not 19, has the near-perfect correlation reported in the draft.

In [ ]:
listener_rows = load_rows('listener_similarity', [
    'listener_id', 'n_ratings', 'speaker_accent_srcc', 'identical_percent', 'n_identical'])
if len({row['listener_id'] for row in listener_rows}) != len(listener_rows):
    raise ValueError('Duplicate listener IDs')
def listener_order(row):
    rho = float(row['speaker_accent_srcc'])
    return (not np.isfinite(rho), -rho if np.isfinite(rho) else 0, row['listener_id'])
listener_rows.sort(key=listener_order)
listener_rho = np.array([float(row['speaker_accent_srcc']) for row in listener_rows])
listener_equal = numbers(listener_rows, 'identical_percent')
listener_counts = numbers(listener_rows, 'n_ratings')
if (listener_counts <= 0).any():
    raise ValueError('Rating counts must be positive')
np.testing.assert_allclose(listener_equal, 100 * numbers(listener_rows, 'n_identical') / listener_counts)
listener_summary = {
    'n_listeners': len(listener_rows), 'n_ratings': int(listener_counts.sum()), 'split': 'train',
    'listeners': [{
        'listener_id': row['listener_id'], 'n_ratings': int(float(row['n_ratings'])),
        'speaker_accent_srcc': float(row['speaker_accent_srcc']) if np.isfinite(float(row['speaker_accent_srcc'])) else None,
        'identical_percent': float(row['identical_percent']),
    } for row in listener_rows],
}
show_table(['Listener', 'Rated pairs', 'Speaker--accent SRCC', 'Identical scores (%)'], [
    [r['listener_id'], r['n_ratings'], r['speaker_accent_srcc'], r['identical_percent']]
    for r in listener_summary['listeners']
])

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.0),
                       layout='constrained')
positions = np.arange(len(listener_rows))
finite_rho = listener_rho[np.isfinite(listener_rho)]
has_negative = len(finite_rho) and finite_rho.min() < 0
for y in positions[::2]:
    ax.axhspan(y - 0.5, y + 0.5, color='#F3F5F7', zorder=0)
ax.scatter(listener_rho, positions, s=30, color=SPEAKER_COLOR, zorder=3)
for y, value in zip(positions, listener_rho):
    ax.text(value + 0.025 if np.isfinite(value) else 0.02, y,
            f'{value:.3f}' if np.isfinite(value) else 'undefined',
            va='center', fontsize=ANNOTATION_SIZE)
ax.set(xlim=(-1.02 if has_negative else -0.02, 1.18),
       xticks=[-1, -0.5, 0, 0.5, 1] if has_negative else [0, 0.25, 0.5, 0.75, 1])
ax.set_xlabel(bold('Speaker--accent Spearman correlation'), fontsize=18)
ax.grid(axis='x', linestyle='--', alpha=0.35)
ax.set_axisbelow(True)
ax.tick_params(axis='y', length=0)
style_axes(ax, numeric_y=False)
ax.set_yticks(positions, [bold(r['listener_id']) for r in listener_rows])
ax.tick_params(axis='y', labelsize=12, pad=2)
ax.set_ylabel(bold('Listener ID'), fontsize=18)
ax.set_ylim(len(listener_rows) - 0.5, -0.5)
save_figure(fig, 'listener_similarity', listener_summary)


## 3. Do listeners agree less about accent similarity?

The same **2,800 training pairs** in both dimensions, with all listeners retained.
Each CSV row contains the sample SD (`ddof=1`) of the ratings for one pair.
Panel (a) compares empirical cumulative distributions; panel (b) shows the
paired difference, accent SD minus speaker SD. Every pair has equal weight.

This measures dispersion on a bounded 1--5 scale, not chance-corrected agreement.
The results are descriptive, without confidence intervals or a causal claim
about accent prediction difficulty. The central histogram bin includes small
nonzero differences; the percentage of exact ties is calculated separately.

In [ ]:
disagreement_rows = load_rows('listener_disagreement', [
    'wav_a_path', 'wav_b_path', 'n_ratings', 'speaker_sd', 'accent_sd', 'accent_minus_speaker_sd'])
unique_pairs(disagreement_rows)
speaker_sd = numbers(disagreement_rows, 'speaker_sd')
accent_sd = numbers(disagreement_rows, 'accent_sd')
if (speaker_sd < 0).any() or (accent_sd < 0).any():
    raise ValueError('Standard deviations must be nonnegative')
sd_difference = accent_sd - speaker_sd
np.testing.assert_allclose(sd_difference, numbers(disagreement_rows, 'accent_minus_speaker_sd'), atol=1e-12)
equal_sd = np.isclose(sd_difference, 0, atol=1e-12, rtol=0)
disagreement_summary = {
    'n_pairs': len(disagreement_rows), 'split': 'train', 'sd_ddof': 1,
    'n_ratings': int(numbers(disagreement_rows, 'n_ratings').sum()),
    'speaker_mean_sd': float(speaker_sd.mean()), 'accent_mean_sd': float(accent_sd.mean()),
    'mean_accent_minus_speaker_sd': float(sd_difference.mean()),
    'accent_higher_percent': float(100 * np.mean((sd_difference > 0) & ~equal_sd)),
    'equal_percent': float(100 * equal_sd.mean()),
    'accent_lower_percent': float(100 * np.mean((sd_difference < 0) & ~equal_sd)),
}
show_table(['Statistic', 'Value'], list(disagreement_summary.items()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout='constrained')
sd_limit = max(2.5, np.ceil(max(speaker_sd.max(), accent_sd.max()) * 2) / 2)
gap_limit = max(2.0, np.ceil(np.max(np.abs(sd_difference)) * 2) / 2)
for values, color, label in [(speaker_sd, SPEAKER_COLOR, 'Speaker'), (accent_sd, ACCENT_COLOR, 'Accent')]:
    unique, counts = np.unique(values, return_counts=True)
    axes[0].step(np.r_[0, unique, sd_limit], np.r_[0, 100 * np.cumsum(counts) / len(values), 100],
                 where='post', linewidth=2, color=color,
                 label=bold(f'{label} (mean SD = {values.mean():.3f})'))
axes[0].set(xlim=(0, sd_limit), ylim=(0, 103))
axes[0].set_title(bold('(a) Listener disagreement across pairs'))
axes[0].set_xlabel(bold('Within-pair rating standard deviation'))
axes[0].set_ylabel(bold('Pairs at or below this SD (%)'))
axes[0].legend(loc='lower right', frameon=False)

bins = np.arange(-gap_limit - 0.05, gap_limit + 0.06, 0.1)
axes[1].hist(sd_difference, bins=bins, weights=np.full(len(sd_difference), 100 / len(sd_difference)),
             color='#8F9CAA', edgecolor='white', linewidth=0.5)
axes[1].axvline(0, color='#333333', linewidth=1)
axes[1].axvline(sd_difference.mean(), color=ACCENT_COLOR, linestyle='--', linewidth=2)
axes[1].set_xlim(-gap_limit - 0.05, gap_limit + 0.05)
axes[1].set_title(bold('(b) Difference for the same pair'))
axes[1].set_xlabel(bold('Accent SD - speaker SD'))
axes[1].set_ylabel(bold('Pairs (%)'))
note = '\n'.join([
    bold(f'Mean difference: {sd_difference.mean():+.3f}'),
    bold(f"Accent higher: {disagreement_summary['accent_higher_percent']:.2f}%"),
    bold(f"Equal: {disagreement_summary['equal_percent']:.2f}%"),
    bold(f"Accent lower: {disagreement_summary['accent_lower_percent']:.2f}%"),
])
axes[1].text(0.03, 0.96, note, transform=axes[1].transAxes, va='top', fontsize=ANNOTATION_SIZE,
             bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.9})
for ax in axes:
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    ax.set_axisbelow(True)
    style_axes(ax)
save_figure(fig, 'listener_disagreement', disagreement_summary)

## 4. Does the model recover differences between speaker and accent similarity?

For the same **600 test pairs**, compare human `accent MOS - speaker MOS` with
the corresponding predicted difference. Equal axis scales and the diagonal
show exact recovery. This section retains the hexagonal density representation
of the existing attribute-differences figure; the human-versus-model section
above uses scatter points.

All pairs contribute to SRCC and MAE. Direction agreement excludes the 140
exact human ties only: it uses 460 pairs. A zero predicted gap on a non-tied
human pair counts as incorrect. Differences are rounded to 12 decimal places
to avoid artificial splitting of mathematical ties. Small human gaps remain
noisy estimates, not certain perceptual distinctions.

In [ ]:
difference_rows = load_rows('attribute_differences', [
    'wav_a_path', 'wav_b_path', 'human_speaker', 'human_accent', 'model_speaker', 'model_accent',
    'human_accent_minus_speaker', 'model_accent_minus_speaker'])
unique_pairs(difference_rows)
human_gap = np.round(numbers(difference_rows, 'human_accent') - numbers(difference_rows, 'human_speaker'), 12)
model_gap = np.round(numbers(difference_rows, 'model_accent') - numbers(difference_rows, 'model_speaker'), 12)
np.testing.assert_allclose(human_gap, numbers(difference_rows, 'human_accent_minus_speaker'), atol=1e-12)
np.testing.assert_allclose(model_gap, numbers(difference_rows, 'model_accent_minus_speaker'), atol=1e-12)
nonzero_gap = human_gap != 0
direction_correct = np.sign(human_gap[nonzero_gap]) == np.sign(model_gap[nonzero_gap])
difference_summary = {
    'n_pairs': len(difference_rows), 'split': 'test', 'model_fit': 'train+dev',
    'gap_srcc': srcc(human_gap, model_gap),
    'gap_mae': float(np.abs(model_gap - human_gap).mean()),
    'zero_gap_baseline_mae': float(np.abs(human_gap).mean()),
    'human_mean_absolute_gap': float(np.abs(human_gap).mean()),
    'model_mean_absolute_gap': float(np.abs(model_gap).mean()),
    'human_equal_score_pairs': int((~nonzero_gap).sum()),
    'direction_agreement_denominator': int(nonzero_gap.sum()),
    'direction_agreement_n_correct': int(direction_correct.sum()),
    'direction_agreement_percent': float(100 * direction_correct.mean()) if nonzero_gap.any() else None,
}
show_table(['Statistic', 'Value'], list(difference_summary.items()))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7), layout='constrained')
limit = max(0.5, np.ceil(max(np.max(np.abs(human_gap)), np.max(np.abs(model_gap))) * 2) / 2)
density = ax.hexbin(human_gap, model_gap, gridsize=24, extent=(-limit, limit, -limit, limit),
                    mincnt=1, cmap='viridis', linewidths=0)
assert int(density.get_array().sum()) == len(difference_rows)
max_count = max(2, float(density.get_array().max()))
density.set_norm(LogNorm(vmin=1, vmax=max_count))
ax.plot([-limit, limit], [-limit, limit], '--', color='#888888', linewidth=2,
        label=bold('Exact recovery (y = x)'))
ax.axhline(0, color='#999999', linewidth=0.8, zorder=0)
ax.axvline(0, color='#999999', linewidth=0.8, zorder=0)
ax.set(xlim=(-limit - 0.05, limit + 0.05), ylim=(-limit - 0.05, limit + 0.05), aspect='equal')
ax.set_title(bold('Weak-16: recovery of attribute differences'))
ax.set_xlabel(bold('Human MOS difference (accent - speaker)'))
ax.set_ylabel(bold('Predicted difference (accent - speaker)'))
ax.legend(loc='upper left', frameon=False)
rho = difference_summary['gap_srcc']
agreement = difference_summary['direction_agreement_percent']
rho_label = f'{rho:.3f}' if rho is not None else 'undefined'
agreement_label = f'{agreement:.1f}%' if agreement is not None else 'undefined'
note = '\n'.join([
    bold(f"{len(difference_rows)} pairs; gap SRCC = {rho_label}; MAE = {difference_summary['gap_mae']:.3f}"),
    bold(f"Direction agreement = {agreement_label} ({difference_summary['direction_agreement_denominator']} non-tied pairs)"),
])
ax.text(0, -0.24, note, transform=ax.transAxes, va='top', fontsize=ANNOTATION_SIZE)
style_axes(ax)
colorbar = fig.colorbar(density, ax=ax, shrink=0.8, pad=0.025)
colorbar.set_label(bold('Pairs per hexagon (log scale)'), fontsize=LABEL_SIZE)
colorbar.set_ticks([value for value in [1, 2, 5, 10, 20, 50, 100, 200, 500] if value <= max_count])
colorbar.ax.yaxis.set_major_formatter(FuncFormatter(lambda value, position: bold(f'{value:g}')))
colorbar.ax.tick_params(labelsize=TICK_SIZE)
colorbar.minorticks_off()
save_figure(fig, 'attribute_differences', difference_summary)

## Download the generated figures

This ZIP contains only the outputs generated in this notebook, organized by
analysis: PDF, PNG, SVG, and a statistics JSON file for each completed section.
The original plotting CSVs and source figures are not modified. In Colab this
cell starts one ZIP download; locally it displays a download link instead.

In [ ]:
generated = list(OUTPUT_DIR.glob('*/*.pdf'))
if not generated:
    raise RuntimeError('Run at least one plotting section before downloading.')
archive = shutil.make_archive(str(OUTPUT_DIR.parent / 'voicemos_latex_figures'), 'zip', OUTPUT_DIR)
print(f'Packaged {len(generated)} figure(s): {archive}')
if IN_COLAB:
    files.download(archive)
else:
    display(FileLink(archive))